In [ ]:
import pandas as pd

labels = pd.read_csv("../data/train_labels.csv")

# Sort alphabetically for a consistent, reproducible mapping
class_names = sorted(labels['class'].unique())
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

labels['label'] = labels['class'].map(class_to_idx)

print(class_to_idx)
labels.head()

In [ ]:
labels['filename'] = labels['image_id'].apply(lambda x: f"{x:05d}.jpg")
labels.head()

In [8]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    labels,
    test_size=0.2,
    stratify=labels['label'],
    random_state=42
)

print(f"Train: {len(train_df)} images")
print(f"Val:   {len(val_df)} images")

# Confirm class proportions match between the two sets
print("\nTrain class proportions:")
print(train_df['class'].value_counts(normalize=True).sort_index())
print("\nVal class proportions:")
print(val_df['class'].value_counts(normalize=True).sort_index())

Train: 50928 images
Val:   12732 images

Train class proportions:
class
CatEye      0.218092
CatFace     0.109056
CatMouth    0.109056
CatNose     0.109036
DogEye      0.181904
DogFace     0.090952
DogMouth    0.090952
DogNose     0.090952
Name: proportion, dtype: float64

Val class proportions:
class
CatEye      0.218112
CatFace     0.109017
CatMouth    0.109017
CatNose     0.109095
DogEye      0.181904
DogFace     0.090952
DogMouth    0.090952
DogNose     0.090952
Name: proportion, dtype: float64


In [9]:
import tensorflow as tf

IMG_SIZE = 224
IMG_DIR = "../data/train/train"

def load_and_preprocess(filename, label):
    path = IMG_DIR + "/" + filename
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0  # normalize to [0,1]
    return image, label

# Quick test on a single file
test_img, test_label = load_and_preprocess(train_df.iloc[0]['filename'], train_df.iloc[0]['label'])
print(test_img.shape, test_img.dtype)
print("min/max pixel value:", float(tf.reduce_min(test_img)), float(tf.reduce_max(test_img)))

(224, 224, 3) <dtype: 'float32'>
min/max pixel value: 0.0 1.0


In [10]:
BATCH_SIZE = 32

def make_dataset(df, shuffle=False):
    filenames = df['filename'].values
    labels_arr = df['label'].values
    ds = tf.data.Dataset.from_tensor_slices((filenames, labels_arr))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=42)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)

# Quick check: pull one batch and confirm shapes
for images, batch_labels in train_ds.take(1):
    print("Batch images shape:", images.shape)
    print("Batch labels shape:", batch_labels.shape)
    print("Sample labels:", batch_labels.numpy()[:10])

Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Sample labels: [7 6 0 2 3 5 6 6 1 7]


In [11]:
overlap = set(train_df['filename']) & set(val_df['filename'])
print(f"Overlapping filenames between train and val: {len(overlap)}")

Overlapping filenames between train and val: 0


In [12]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomBrightness(0.1, value_range=(0.0, 1.0)),
    tf.keras.layers.RandomContrast(0.1),
])

def augment(image, label):
    image = data_augmentation(image, training=True)
    image = tf.clip_by_value(image, 0.0, 1.0)  # safety net against any small overshoot
    return image, label

train_ds_aug = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

for images, labels_batch in train_ds.take(1):
    aug_images, _ = augment(images, labels_batch)
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i in range(5):
        axes[0, i].imshow(images[i])
        axes[0, i].set_title(f"Original: {idx_to_class[labels_batch[i].numpy()]}")
        axes[0, i].axis('off')
        axes[1, i].imshow(aug_images[i])
        axes[1, i].set_title("Augmented")
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

NameError: name 'plt' is not defined

In [13]:
import os

TEST_DIR = "../data/test/test"
sample_sub = pd.read_csv("../data/sample_submission.csv")

# Build the same zero-padded filename format used for train
sample_sub['filename'] = sample_sub['image_id'].apply(lambda x: f"{x:05d}.jpg")

print(sample_sub.shape)
sample_sub.head()

(29443, 3)


,image_id,class,filename
0,1,DogEye,00001.jpg
1,2,CatNose,00002.jpg
2,3,CatNose,00003.jpg
3,4,DogFace,00004.jpg
4,5,DogMouth,00005.jpg


In [14]:
def load_test_image(filename):
    path = TEST_DIR + "/" + filename
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    return image

test_filenames = sample_sub['filename'].values

test_ds = tf.data.Dataset.from_tensor_slices(test_filenames)
test_ds = test_ds.map(load_test_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

# Quick check
for batch in test_ds.take(1):
    print("Test batch shape:", batch.shape)

Test batch shape: (32, 224, 224, 3)
